In [1]:
import os
import time
import urllib.request
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
import pandas as pd
import numpy as np
from pytesseract import Output
import pytesseract
import cv2
#import matplotlib.pyplot as plt
from PIL import Image

from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import urllib.request

from selenium.webdriver.support.ui import WebDriverWait 
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import NoSuchElementException

In [2]:
def initialize_driver():
    chrome_options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(service=Service('D:\chromedriver-win32\chromedriver.exe'), options=chrome_options)
    return driver

def get_theater_links(driver, query):
    driver.get(query)
    time.sleep(1)
    number =160

    addresslist = []

    for i in range(1, int(number) + 1):
        xpath = f"/html/body/table/tbody/tr[2]/td[3]/div/div/div[2]/div/table/tbody/tr[{i}]/td[1]/a"
        elem = driver.find_element(By.XPATH, xpath).get_attribute("href")
        addresslist.append(elem)

    return addresslist

def scrape_cast(driver, addresslist):
    cast = []

    for lis in addresslist:
        driver.get(lis)
        time.sleep(1)
        
        # 팝업창 닫기
        try:
            close_button = driver.find_element(By.CLASS_NAME, "popupCloseBtn.is-bottomBtn")
            close_button.click()
            time.sleep(0.5)
        except NoSuchElementException:
             pass
        
        title = driver.find_element(By.XPATH, "/html/body/div[1]/div[5]/div[1]/div[2]/div[1]/div/div[1]/h2").text
        age_percentages = {}

        try:
            nav = driver.find_elements(By.CLASS_NAME, "navItem")
            for i in nav:
                if "판매정보" in i.text:
                    i.click()
                    time.sleep(2)

            cast_element = driver.find_element(By.XPATH, "//th[text()='주연']/following-sibling::td/span[@class='moreInfoCast']")
            cast_info = cast_element.text
            cast.append({'Title': title, 'cast_act' : cast_info})

        except Exception as e:
            print(f"Error while processing {title}: {str(e)}")

    return cast

def save_to_csv(data_list, save_path):
    if data_list:
        data = pd.DataFrame(data_list)
        data.to_csv(save_path, index=False)

In [3]:
current_directory = os.getcwd()
save_path = os.path.join(current_directory, "D:/뮤지컬160_배우.csv")

driver = initialize_driver()
addresslist = get_theater_links(driver, "http://ticket.interpark.com/TPGoodsList.asp?Ca=Mus&Sort=3")
scraped_data = scrape_cast(driver, addresslist)
save_to_csv(scraped_data, save_path)
driver.quit()

In [5]:
a=pd.read_csv("D:/뮤지컬160_배우.csv")
a

,Title,cast_act
0,뮤지컬 〈레베카〉 10주년 기념공연,"류정한, 민영기, 에녹, 테이, 신영숙, 옥주현, 리사, 장은아, 김보경, 이지혜,..."
1,태양의서커스 〈루치아〉,해당없음
2,뮤지컬 〈오페라의 유령〉 - 서울,"조승우, 최재림, 김주택, 전동석, 손지수, 송은혜, 송원근, 황건하, 윤영석, 이..."
3,뮤지컬 ＇벤허＇,"박은태, 신성록, 규현, 이지훈, 박민성, 서경수, 윤공주, 이정화, 최지혜, 이정..."
4,뮤지컬 〈멤피스〉,"박강현, 고은성, 이창섭, 정선아, 유리아, 손승연, 최민철, 심재현, 최정원, 류..."
...,...,...
155,콘서트뮤지컬 〈위대한 귀향〉,해당없음
156,태양의서커스 〈루치아〉 - 부산,해당없음
157,트롯 마당놀이 〈흥놀부뎐〉,"강민주, 강동현, 이윤경, 고종승"
158,페인터즈,해당없음


In [6]:
a.to_excel('D:/뮤지컬160_배우.xlsx')